In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt
from shapely.geometry import box

In [3]:
sal = gpd.read_file("data/2011_census/2011_census/ea_sal_kzn_gp.shp")

wards = gpd.read_file("data/2023_census/2023_census/SA_Wards2020.shp")

wards_with_pop = gpd.read_file("data/2023_census/PP_Population Group_27-10-2025.csv")

DataSourceError: data/2011_census/2011_census/ea_sal_kzn_gp.shp: No such file or directory

In [ ]:
pd.set_option('display.float_format', '{:,.0f}'.format)

pd.set_option('display.max_columns', 100)

In [ ]:
sal.columns

In [ ]:
wards.columns

In [ ]:
wards_with_pop.columns

In [ ]:
wards_with_pop = wards_with_pop.rename(columns = {"Ward_Code" : "WardID"})

In [ ]:
wards = wards.merge(
    wards_with_pop[['WardID', 'Total',]],
    on='WardID',
    how='left'
)

In [ ]:
wards = wards[wards['Province'].isin(['Gauteng', 'KwaZulu-Natal'])].copy()

In [ ]:
sal['population'].isna().sum()

In [ ]:
wards

In [ ]:
sal

In [ ]:
sal_with_ward = gpd.read_file("data/sal_w_ward_new/sal_w_ward_new.shp")

In [ ]:
pd.set_option('display.float_format', '{:,.2f}'.format)

In [ ]:
sal_with_ward=sal_with_ward.rename(columns={'census_war': 'WardID'})

In [ ]:
sal_with_ward = sal_with_ward.merge(
    wards[['WardID', 'Total']],
    on='WardID',
    how='left'
)

In [ ]:
sal_wards= sal_with_ward[['WardID', 'EA_CODE', 'sal2011_po', "Total", 'EA_GTYPE', 'EA_TYPE', 'F4_class', 'num_houses', 
                         'Black_Afri', 'White', 'Coloured', 'Indian_or', 'Other',    ]]

In [ ]:
sal_wards=sal_wards.rename(columns={'sal2011_po': 'sal2011_pop',
                            'Total':'ward2023_pop',
                                'F4_class': 'econ_status',
                                'num_houses': 'houses2011'})

In [ ]:
sal_wards['EA_TYPE'] = sal_wards['EA_TYPE'].str.replace(r'_\*$', '', regex=True)
sal_wards['EA_TYPE'] = sal_wards['EA_TYPE'].str.replace(
    'Smallholdings', 'Small holdings'
)

In [ ]:
sal_wards.loc[sal_wards['sal2011_pop'] == 0, 'sal2011_pop'] = sal_wards['houses2011']*3


sal_wards['area_km2'] = sal_with_ward.geometry.area / 1e6

sal_wards['sal_dense'] = (
    sal_wards['sal2011_pop'].astype(float) /
    sal_wards['area_km2'].astype(float)
)
sal_wards['log_density'] = np.log1p(sal_wards['sal_dense'])

In [ ]:
sal_wards

In [ ]:
sal_wards['log_density'].describe()

In [ ]:
sal_wards['ward2023_pop'] = pd.to_numeric(sal_wards['ward2023_pop'], errors='coerce')
sal_wards['sal2011_pop']= pd.to_numeric(sal_wards['sal2011_pop'], errors='coerce')

In [ ]:
sal_wards[sal_wards.duplicated(subset='EA_CODE', keep=False)]
sal_wards = sal_wards.drop_duplicates(subset='EA_CODE', keep='first')

In [ ]:
#Calculating proportion of SAL pop within Ward pop

ward2011_sum = sal_wards.groupby('WardID', as_index=False)['sal2011_pop'].sum()
ward2011_sum = ward2011_sum.rename(columns={'sal2011_pop': 'ward2011_sum'})
sal_wards = sal_wards.merge(
    ward2011_sum,
    on='WardID',
    how='left'
)


sal_wards['share2011']=sal_wards['sal2011_pop']/sal_wards['ward2011_sum']

In [ ]:
sal_wards['share2011'].describe()

In [ ]:
#SAL weights based on density and area

In [ ]:
# gtype_growth = sal_wards.groupby('EA_TYPE').agg(
#     pop2011=('ward2011_sum','sum'),
#     pop2023=('ward2023_pop','sum')
# ).reset_index()


# gtype_growth['growth_ratio'] = gtype_growth['pop2023'] / gtype_growth['pop2011']

# weights_dict = dict(zip(gtype_growth['EA_TYPE'], gtype_growth['growth_ratio']))
# sal_wards['gweight'] = sal_wards['EA_TYPE'].map(weights_dict)

In [ ]:
# sal_wards['gweight'].describe()

In [ ]:
sal_wards['dasym_weight']= sal_wards['share2011']*sal_wards['log_density']
sal_wards['dasym_weight'] = sal_wards['dasym_weight']/sal_wards.groupby('WardID')['dasym_weight'].transform('sum')

In [ ]:
sal_wards['dasym_weight'].describe()

In [ ]:
#Estimating 2023 SAL population based on weights and 2023 ward level counts
sal_wards['sal2023_est'] = sal_wards['dasym_weight']* sal_wards['ward2023_pop']

In [ ]:
#calculating growth rate using 2011 counts and 2023 estimate

sal_wards['growth_rate'] = ((sal_wards['sal2023_est'] / sal_wards['sal2011_pop'])**(1/12)) - 1

In [ ]:
#Readjusting 2026 predictions to stay within total population margins and account for sprawl from one SAL into another
#wards['Total'] = pd.to_numeric(wards['Total'], errors='coerce')
#totals2011 = sal_wards['sal2011_pop'].sum()
#totals2023 = wards['Total'].sum()

#annual_growth = ((totals2023 / totals2011)**(1/12)) 

#target_2026 = totals2023 * (( annual_growth)**3)

#sal_wards['share2023'] = sal_wards['sal2023_est'] / totals2023

#sal_wards['sal2026_pred'] = sal_wards['share2023'] * target_2026

In [ ]:
sal_wards['ward2023_pop'].describe()

In [ ]:
sal_wards['sal2023_est'].describe()

In [ ]:
#sal_wards['sal2026_pred'].describe()

In [ ]:
#sal_wards['sal2026_pred'].sum()

In [ ]:
sal_wards['sal2023_est'].sum()

In [ ]:
wards['ward2023_pop'] = pd.to_numeric(wards['Total'], errors='coerce')

In [ ]:
wards['ward2023_pop'].sum()

In [ ]:
sal_wards['sal2023_est'].sum()-wards['ward2023_pop'].sum() 

In [ ]:
sal_wards[['sal2011_pop','sal2023_est', ]].sum()

In [ ]:
sal_wards['EA_CODE'] = sal_wards['EA_CODE'].astype('Int64')   # or str if needed

In [ ]:
sal_wards

In [ ]:
import tabulate
print(sal_wards.head().to_markdown(index=False, floatfmt=",.2f"))

In [ ]:
gtype_summary = sal_wards.groupby('EA_TYPE').agg(
    pop2011=('sal2011_pop','sum'),
    pop2023=('sal2023_est','sum'),
).reset_index()

gtype_summary['growth_rate_2011_2023'] = (
    ((gtype_summary['pop2023'] / gtype_summary['pop2011']) ** (1/12) - 1)*100
)

In [ ]:
sal_wards['sal_dense'].describe()

In [ ]:
gtype_summary

In [ ]:
#print(gtype_summary.to_markdown(index=False, floatfmt=",.2f"))

In [ ]:
sal_map = sal.merge(
    sal_wards,
    on='EA_CODE',
    how='left'
)

In [ ]:
sal_wards['sal2023_est'].describe()

In [ ]:
import mapclassify
import matplotlib.pyplot as plt
import matplotlib as mpl

fig, axes = plt.subplots(1, 2, figsize=(16,10))

provinces = sal_map['PR_NAME'].unique()

# consistent bins
bin_edges = [0, 500,1000, 2500, 5000, 10000, 15000]

cmap = mpl.cm.viridis_r

for i, prov in enumerate(provinces):

    subset = sal_map[sal_map['PR_NAME'] == prov].copy()
    subset = subset[subset['sal2023_est'].notna()]  # drop missing

    subset.plot(
        column='sal2023_est',
        cmap=cmap,
        scheme='UserDefined',
        classification_kwds={'bins': bin_edges},
        legend=False,
        ax=axes[i]
    )

    axes[i].set_title(prov, fontsize=14)
    axes[i].axis('off')

# ---- shared colorbar (fixed mapping) ----
norm = mpl.colors.BoundaryNorm(bin_edges, ncolors=cmap.N)
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

cbar_ax = fig.add_axes([0.48, 0.25, 0.02, 0.5])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label("Population")

plt.suptitle("SAL 2023 Estimate Count")

output_file = "images/newpred/pop2023estimate.png"
plt.savefig(output_file, dpi=300, bbox_inches='tight')

plt.show()

print(f"Map saved to {output_file}")

In [ ]:
sal_map['dense_2023']= sal_map['sal2023_est']/sal_map['area_km2']

In [ ]:
sal_map['dense_2023'].describe()

In [ ]:
sal_map['log_density'] = np.log1p(sal_map['dense_2023'])

In [ ]:
import mapclassify
import matplotlib.pyplot as plt
import matplotlib as mpl
fig, axes = plt.subplots(1, 2, figsize=(16,10))

provinces = sal_map['PR_NAME'].unique()

for i, prov in enumerate(provinces):
    
    subset = sal_map[sal_map['PR_NAME'] == prov].copy()
    
    # compute quintile breaks
    subset.plot(
        column='log_density',
        cmap='viridis_r',
        legend=False,
        scheme='Quantiles',
        k=6,
        ax=axes[i]
    )
        
    axes[i].set_title(prov, fontsize=14)
    axes[i].axis('off')

# 🔥 Create shared colorbar in the middle
norm = mpl.colors.Normalize(
    vmin=sal_map['log_density'].min(),
    vmax=sal_map['log_density'].max()
)
sm = mpl.cm.ScalarMappable(cmap='viridis_r', norm=norm)
sm.set_array([])

# [left, bottom, width, height] → tweak these numbers if needed
cbar_ax = fig.add_axes([0.48, 0.25, 0.02, 0.5])  

cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label("Population Density (normalized) per km sq")
plt.suptitle("SAL 2023 Density (Log)")
output_file = "images/newpred/popdensity.png"
plt.savefig(output_file, dpi=300, bbox_inches='tight')

plt.show()

print(f"Map saved to {output_file}")

In [ ]:
sal_wards.to_csv('data/pop_pred_final.csv', index=False)

In [ ]:
sal_map = sal_map.to_crs(sal_with_ward.crs)

In [ ]:
import mapclassify
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Rectangle

# -------------------------
# FILTER DATA
# -------------------------
gauteng = sal_map[sal_map['PR_NAME'].str.contains("Gauteng", case=False, na=False)].copy()

jhb = gauteng[gauteng['MN_NAME'].str.contains("Johannesburg", case=False, na=False)]

# -------------------------
# FIGURE (Joburg bigger)
# -------------------------
fig, axes = plt.subplots(
    1, 2,
    figsize=(16,10),
    gridspec_kw={'width_ratios': [1, 2]}  # Joburg panel bigger
)

# -------------------------
# LEFT: GAUTENG (context)
# -------------------------
gauteng.plot(
    column='log_density',
    cmap='viridis_r',
    legend=False,
    scheme='Quantiles',
    k=8,
    ax=axes[0]
)

# Add rectangular outline for Johannesburg
xmin, ymin, xmax, ymax = jhb.total_bounds
pad_x = (xmax - xmin) * 0.02  # slight padding
pad_y = (ymax - ymin) * 0.02

rect = Rectangle(
    (xmin - pad_x, ymin - pad_y),
    (xmax - xmin) + 2*pad_x,
    (ymax - ymin) + 2*pad_y,
    linewidth=2,
    edgecolor='red',
    facecolor='none'
)
axes[0].add_patch(rect)

axes[0].set_title("Gauteng (Context)", fontsize=12)
axes[0].axis('off')

# -------------------------
# RIGHT: JOHANNESBURG (zoom)
# -------------------------
jhb.plot(
    column='log_density',
    cmap='viridis_r',
    legend=False,
    scheme='Quantiles',
    k=8,
    ax=axes[1]
)

axes[1].set_title("Johannesburg Population Density ", fontsize=14)
axes[1].axis('off')

# -------------------------
# SHARED COLORBAR
# -------------------------
norm = mpl.colors.Normalize(
    vmin=sal_map['log_density'].min(),
    vmax=sal_map['log_density'].max()
)

sm = mpl.cm.ScalarMappable(cmap='viridis_r', norm=norm)
sm.set_array([])

cbar = fig.colorbar(
    sm,
    ax=axes,
    fraction=0.02,
    pad=0.04
)
cbar.set_label("Population Density (normalized) per km sq")

# -------------------------
# SAVE
# -------------------------
output_file = "images/newpred/gauteng_joburg_sidebyside_rect.png"
plt.savefig(output_file, dpi=300, bbox_inches='tight')

plt.show()
print(f"Map saved to {output_file}")

In [ ]:
sal_wards['sal_dense'].describe()

In [ ]:
print(sal_with_ward.crs)

In [ ]:
print(sal_wards['share2011'].describe().to_markdown(index=False, floatfmt=",.2f"))

In [ ]:
sal_wards[sal_wards['sal_dense'] >100000].sort_values('area_km2')

In [ ]:
cols = ['sal2023_est', 'sal2011_pop', 'growth_rate', 
        'dasym_weight', 'share2011', 'log_density']

summary = sal_wards[cols].describe()

print(gtype_summary.to_markdown(floatfmt=",.3f"))

In [ ]:
print(sal_wards.loc[sal_wards['growth_rate'].idxmin()].to_markdown(index=True, floatfmt=",.2f"))

In [ ]:
sal_wards.loc[sal_wards['sal2023_est'].idxmax()]